# FinD-Generator: Full Example Run

This notebook provides a complete, end-to-end example of using the FinD-Generator pipeline. It covers:

1.  **Configuration**: Set up all model and training parameters in one place.
2.  **Data Collection**: Fetch raw financial and macroeconomic data.
3.  **Data Preprocessing**: Transform, scale, and prepare the data for the model.
4.  **Model Training**: Train the `ConditionalTimeGrad` model.
5.  **Inference & Forecasting**: Generate predictions on new data and visualize the results.
6.  **Scenario Analysis**: Create forecasts based on hypothetical future conditions.

## 1. Setup and Imports

First, we'll import the necessary libraries and add the project's source directory to the Python path to ensure our modules can be found.

In [ ]:
import sys
import os
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add the project directory to the Python path to allow imports from `src`
# This is necessary for the notebook to find the custom modules.
if 'FinD_Generator' not in sys.path:
    sys.path.append('FinD_Generator')

# Import custom modules from the project
from src.data_collector import DataCollector
from src.data_loader import TimeGradDataModule
from src.training.trainer import TimeGradTrainer
from inference import TimeGradPredictor, plot_forecast

## 2. Configuration

All key parameters for the pipeline are defined here. You can easily modify them to experiment with different settings. For a quick test run, we'll use a small number of epochs.

In [ ]:
config = {
    # Data Parameters
    'seq_len': 120,         # Historical window length
    'horizon': 30,          # Forecast horizon

    # Model Parameters
    'diff_steps': 100,      # Number of diffusion steps
    'beta_end': 0.1,        # Final beta value for diffusion noise schedule
    'residual_layers': 8,   # Number of residual layers in the WaveNet backbone
    'residual_channels': 8, # Number of channels in the residual layers

    # Training Parameters
    'epochs': 10,           # Number of training epochs (set low for a quick run)
    'batch_size': 64,       # Batch size
    'lr': 1e-4,             # Initial learning rate
    'max_lr': 1e-3,         # Maximum learning rate for OneCycleLR scheduler
    'weight_decay': 1e-6,   # Optimizer weight decay
    'clip_grad': 1.0,       # Gradient clipping threshold

    # System & Checkpointing
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42,
    'save_dir': 'notebook_checkpoints', # Directory to save model checkpoints
    'save_freq': 5,         # Save a checkpoint every N epochs
}

print(f"Using device: {config['device']}")
print(f"Checkpoints will be saved in: {config['save_dir']}")

## 3. Data Collection

Here, we use the `DataCollector` to fetch all required time series data (target asset, market index, and macroeconomic indicators) from online sources like Yahoo Finance and FRED. The data will be saved locally in the `FinD_Generator/data/raw` directory.

In [ ]:
print("--- Step 1: Kicking off Data Collection ---")
collector = DataCollector()
data_dict = collector.collect_all_data(save=True)

print("\nData collection complete. Collected dataframes:")
for name, df in data_dict.items():
    print(f"- {name}: {df.shape[0]} rows")

print("\nExample: 'target' dataframe head:")
display(data_dict['target'].head())

## 4. Data Preprocessing and Loading

The `TimeGradDataModule` handles the complex preprocessing pipeline. It takes the raw dataframes, applies transformations (denoising, returns, etc.), merges them, creates regime and calendar features, splits the data chronologically, and fits scalers/PCA on the training set to prevent data leakage.

In [ ]:
print("--- Step 2: Initializing Data Preprocessing ---")
dm = TimeGradDataModule(
    data_dict=data_dict,
    seq_len=config['seq_len'],
    forecast_horizon=config['horizon'],
    batch_size=config['batch_size'],
    device=config['device']
)

# This single call runs the entire preprocessing and splitting pipeline
dm.preprocess_and_split()

# This builds the PyTorch Dataset objects
dm.build_datasets()

print("\n--- Data Module Setup Complete ---")
print(f"Train samples: {len(dm.train_set)}")
print(f"Validation samples: {len(dm.val_set)}")
print(f"Test samples: {len(dm.test_set)}")

# Let's inspect a single batch to verify the output shapes
train_loader = dm.train_dataloader()
sample_batch = next(iter(train_loader))

print("\n--- Sample Batch Shapes ---")
for key, tensor in sample_batch.items():
    print(f"{key:>15s}: {str(list(tensor.shape))}")

## 5. Model Training

The `TimeGradTrainer` class encapsulates the entire training process. We initialize it with our configuration and simply call the `.run()` method. It will automatically handle data preparation, model initialization, the training loop, validation, and checkpointing.

In [ ]:
print("--- Step 3: Initializing Model Trainer ---")

# The trainer class uses the config dictionary to set up the entire training process.
trainer = TimeGradTrainer(**config)

# This single call will prepare data, build the model, and run the full training and validation loop.
trainer.run()

## 6. Inference and Forecasting

After training, we can use the `TimeGradPredictor` to load our best model and generate forecasts. We'll perform two types of inference:

1.  **Ground Truth Forecast**: Predict the future for a sample from the validation set and compare it against the actual outcome.
2.  **Scenario Forecast**: Create a hypothetical future by modifying conditioning variables (e.g., assuming a high-volatility market) and see how the model's forecast changes.

### 6.1. Load the Trained Model

We initialize the predictor, which loads the data pipeline and the best-performing model checkpoint saved during training.

In [ ]:
print("--- Step 4: Initializing Inference Predictor ---")
best_model_path = os.path.join(config['save_dir'], 'best_model.pt')

if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Model checkpoint not found at {best_model_path}. Please ensure training was successful.")

predictor = TimeGradPredictor(
    checkpoint_path=best_model_path,
    device=config['device']
)

### 6.2. Generate and Plot a Ground Truth Forecast

Let's pick a sample from the validation set and generate a probabilistic forecast. The plot will show the historical data, the true future values, the model's mean forecast, and the 10%-90% quantile range.

In [ ]:
sample_idx_to_predict = 50 # Pick a sample from the validation set
num_mc_samples = 200       # Number of Monte Carlo samples for the forecast

print(f"\n--- Generating forecast for validation sample index: {sample_idx_to_predict} ---")
predictions, x_hist, x_future_true = predictor.predict(
    sample_idx=sample_idx_to_predict, 
    num_samples=num_mc_samples
)

# Define path for the output plot
output_dir = "FinD_Generator/image/graph"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"notebook_forecast_sample_{sample_idx_to_predict}.png")

# Plot the results
plot_forecast(
    predictions, x_hist, x_future_true,
    title=f"Forecast for Ground Truth (Sample: {sample_idx_to_predict})",
    output_path=output_path,
    is_scenario=False
)

# Display the plot in the notebook
plt.imshow(plt.imread(output_path))
plt.axis('off')
plt.show()

### 6.3. Generate and Plot a Scenario Forecast

Now for the exciting part: let's define a custom scenario. We'll use the same historical data as the base but override some of the conditioning features to see how the model reacts. For example, let's simulate a "High Volatility & Bear Market" scenario.

In [ ]:
# Define the scenario by overriding conditioning features.
# You can find available feature names from the predictor's data module:
# print("Dynamic cols:", predictor.dm.cond_dynamic_cols)
# print("Static cols:", predictor.dm.cond_static_cols)

scenario = {
    "name": "High Volatility & Bear Market",
    "dynamic_overrides": {
        # Set VIX to a constant high value for the forecast horizon
        "daily_vix_daily_scaled": [2.5] * config['horizon'] 
    },
    "static_overrides": {
        # Activate the 'bear' and 'high_vol' regimes
        "market_regime_bear": 1.0,
        "market_regime_bull": 0.0,
        "market_regime_sideways": 0.0,
        "vol_regime_high_vol": 1.0,
        "vol_regime_normal_vol": 0.0
    }
}

print(f"\n--- Generating forecast for scenario: '{scenario['name']}' ---")
scenario_preds, scenario_hist, _ = predictor.predict_scenario(
    sample_idx=sample_idx_to_predict,
    scenario_data=scenario,
    num_samples=num_mc_samples
)

# Define path for the output plot
scenario_output_path = os.path.join(output_dir, "notebook_scenario_forecast.png")

# Plot the scenario results
plot_forecast(
    scenario_preds, 
    scenario_hist, 
    x_future_true, # We can still plot the original ground truth for comparison
    title=f"Forecast for '{scenario['name']}' (Base Sample: {sample_idx_to_predict})",
    output_path=scenario_output_path,
    is_scenario=True
)

# Display the plot in the notebook
plt.imshow(plt.imread(scenario_output_path))
plt.axis('off')
plt.show()